# **netflix Data Insights**

## Objectives

* perform ETL processes on Netflix Movies and Shows, engineer meaningful features and derive insights through data visualizations.

## Inputs

* Write down which data or information you need to run the notebook 

## Outputs

* Write here which files, code or artefacts you generate by the end of the notebook 

## Additional Comments

* If you have any additional comments that don't fit in the previous bullets, please state them here. 



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Users\\ovrcm\\OneDrive\\Documents\\vscode-projects\\netflix-data-analysis\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'c:\\Users\\ovrcm\\OneDrive\\Documents\\vscode-projects\\netflix-data-analysis'

# import libaries and packages

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

---

### 1. Load the Dataset

In this step, I load the Netflix Movies and TV Shows dataset into a pandas DataFrame.  
This allows me to inspect the structure of the data and begin the ETL (Extract, Transform, Load) process.  
I also display the first few rows to confirm the dataset has loaded correctly.

In [6]:
df = pd.read_csv('../dataset/RawData/netflix_titles.csv')
df.head(5)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


### 2. Inspect Dataset Structure

Here I examine the dataset's structure using `df.info()` and check for missing values using `df.isnull().sum()`.  
This helps identify:
- Data types  
- Columns requiring cleaning  
- Missing or inconsistent data  
- Potential transformations needed before analysis


In [7]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

### 3. Handle Missing Values

The dataset inspection shows several columns with missing values, including `director`, `cast`, `country`, `rating`, and `duration`.  
These missing values are expected because Netflix does not always provide full metadata for every title.

Instead of dropping rows (which would remove useful data and reduce the dataset size), I replace missing values with `"Unknown"`.  
This approach preserves all entries while still allowing meaningful analysis in later steps.

In [8]:
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')
df['rating'] = df['rating'].fillna('Unknown')
df['duration'] = df['duration'].fillna('Unknown')
df.isnull().sum()

show_id          0
type             0
title            0
director         0
cast             0
country          0
date_added      10
release_year     0
rating           0
duration         0
listed_in        0
description      0
dtype: int64

### 4. Convert `date_added` to Datetime Format

The `date_added` column is currently stored as text (`object`).  
To analyse trends over time — such as how Netflix’s catalogue has grown — this column needs to be converted into a proper datetime format.

After converting the column, I also extract a new feature called `year_added`, which will allow me to visualise how many titles were added to Netflix each year.  
This step prepares the dataset for time‑based exploratory analysis.


In [10]:
# Convert date_added to datetime using mixed format inference
df['date_added'] = pd.to_datetime(df['date_added'], format='mixed', errors='coerce')

# Extract year_added for time-based analysis
df['year_added'] = df['date_added'].dt.year

# Preview the results
df[['date_added', 'year_added']].head()


,date_added,year_added
0,2021-09-25,2021.0
1,2021-09-24,2021.0
2,2021-09-24,2021.0
3,2021-09-24,2021.0
4,2021-09-24,2021.0


### 5. Split Genre Information

The `listed_in` column has multiple genres in one string.  
I’m splitting them into a list so I can count genres properly later and make better visualisations.


In [11]:
df['genres'] = df['listed_in'].str.split(', ')
df[['listed_in', 'genres']].head()

,listed_in,genres
0,Documentaries,[Documentaries]
1,"International TV Shows, TV Dramas, TV Mysteries","[International TV Shows, TV Dramas, TV Mysteries]"
2,"Crime TV Shows, International TV Shows, TV Act...","[Crime TV Shows, International TV Shows, TV Ac..."
3,"Docuseries, Reality TV","[Docuseries, Reality TV]"
4,"International TV Shows, Romantic TV Shows, TV ...","[International TV Shows, Romantic TV Shows, TV..."


### 6. Clean Duration Column

The `duration` column has values like "90 min" for movies and "2 Seasons" for TV shows.  
I’m going to split this into two parts:

- one column showing the number (minutes or seasons)
- one column showing whether it’s minutes or seasons

This makes it easier to analyse movie lengths and TV show season counts later.


In [12]:
df['duration_type'] = df['duration'].apply(lambda x: 'Season' if 'Season' in x else 'Minutes')

df['duration_int'] = df['duration'].str.extract('(\d+)').astype(float)

#preview results
df[['duration', 'duration_type', 'duration_int']].head()

<>:3: SyntaxWarning: invalid escape sequence '\d'
<>:3: SyntaxWarning: invalid escape sequence '\d'
C:\Users\ovrcm\AppData\Local\Temp\ipykernel_16524\3216500803.py:3: SyntaxWarning: invalid escape sequence '\d'
  df['duration_int'] = df['duration'].str.extract('(\d+)').astype(float)


,duration,duration_type,duration_int
0,90 min,Minutes,90.0
1,2 Seasons,Season,2.0
2,1 Season,Season,1.0
3,1 Season,Season,1.0
4,2 Seasons,Season,2.0


### 7. EDA: Movies vs TV Shows

Now that the data is cleaned, I’m starting the EDA.  
First, I want to see how many Movies and TV Shows are in the dataset.  
This gives me a quick idea of what type of content Netflix has more of.


# Section 2

Section 2 content

---

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)
